# Twój pipeline: dane dostępne w chwili decyzji

Źródło: albim72/ANALIZA_PYTHON_03_26/data_science_pipeline.py. 20 minut, pokaz lub ćwiczenie dla chętnych. Uproszczono generator do 3000 transakcji, usunięto Faker, Parquet i trening Random Forest. To syntetyczna demonstracja przygotowania cech, nie narzędzie oceny rzeczywistych transakcji.

Uruchom komórkę Start. Cały pakiet musi być rozpakowany. W Colab prześlij ZIP i rozpakuj go przez panel Pliki lub komendę `!unzip nazwa.zip`, a następnie ustaw katalog roboczy na folder pakietu.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'dane').exists() and (ROOT.parent/'dane').exists(): ROOT=ROOT.parent
if not (ROOT/'dane').exists(): raise FileNotFoundError('Rozpakuj cały pakiet i otwórz notebook w folderze zawierającym dane oraz narzedzia.')
sys.path.insert(0,str(ROOT/'narzedzia'))
from swps_tools import sales_quality, aggregate_chunks, to_cents, AmountCapper, make_transactions

## 1. Moment decyzji

Wczytaj syntetyczne transakcje. Decyzja zapada przy autoryzacji transakcji, więc kwota bieżącej transakcji jest znana. refund_after_event powstaje później i nie wolno użyć jej jako cechy. Wskaż też identyfikatory, które nie powinny automatycznie trafiać do modelu.

In [ ]:
transactions=pd.read_csv(ROOT/'dane/transakcje_demo.csv',parse_dates=['created_at'])
features=['age','income','amount','past_chargebacks','device_type']
target='is_fraud'
assert 'refund_after_event' not in features
print(transactions.shape,transactions[features].isna().sum().to_dict())
# Cel tej części: przygotowanie danych. Nie uczymy tu klasyfikatora i nie oceniamy realnego ryzyka finansowego.

## 2. Podział przed uczeniem przekształceń

Posortuj po dacie. Oddziel wcześniejsze 80% wierszy od późniejszych 20%. Ustal, co byłoby inne, gdyby w każdym okresie powracali ci sami klienci.

In [ ]:
transactions=transactions.sort_values('created_at')
cut=int(.8*len(transactions))
train=transactions.iloc[:cut].copy();test=transactions.iloc[cut:].copy()
X_train=train[features];X_test=test[features]
assert train.created_at.max()<test.created_at.min()
print('Train:',len(train),'test:',len(test))

## 3. Próg i imputacja na treningu

Zbuduj pipeline: najpierw AmountCapper uczący kwantyl 0.995 na train, potem ColumnTransformer z medianą i skalowaniem dla liczb oraz OneHotEncoder dla urządzenia. W źródłowym data_science_pipeline.py upper_cap był obliczany na całym df przed split. Ta wersja przenosi uczenie progu do fit na treningu.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
numeric=['age','income','amount','past_chargebacks']
prepare=ColumnTransformer([
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]),['device_type'])])
pipeline=Pipeline([('cap',AmountCapper(.995)),('prepare',prepare)])
A=pipeline.fit_transform(X_train)
B=pipeline.transform(X_test)
train_cap=pipeline.named_steps['cap'].upper_
assert np.isclose(train_cap,X_train.amount.quantile(.995))
print('Próg z train:',train_cap,'macierze:',A.shape,B.shape)

## 4. Kontrola niezależności od testu

Zmień kwoty w kopii testu na skrajne i dodaj nieznaną kategorię urządzenia. Wykonaj wyłącznie transform. Czy parametry progu i imputacji uległy zmianie? Nie porównuj skuteczności modelu, bo go tutaj nie trenujemy.

In [ ]:
saved_cap=pipeline.named_steps['cap'].upper_
saved_median=pipeline.named_steps['prepare'].named_transformers_['num'].named_steps['imputer'].statistics_.copy()
changed=X_test.copy();changed['amount']=1_000_000_000.;changed['device_type']='nowe_urzadzenie'
transformed=pipeline.transform(changed)
assert pipeline.named_steps['cap'].upper_==saved_cap
assert np.array_equal(pipeline.named_steps['prepare'].named_transformers_['num'].named_steps['imputer'].statistics_,saved_median)
print('Transform nie uczy parametrów z testu. Nieznana kategoria obsłużona:',transformed.shape)